<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-07-blob-processing-classification-routing/notebooks/Pfizer_Step2_Contract_Chunk_Routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Route Queries Within a Large Contract PDF
**Pfizer AI Document Intelligence Externship — LlamaIndex chunk-level routing**

One multi-page PDF (`pharma-blob-sample.pdf`), chunked with a sliding window and indexed
with LlamaIndex. Every chunk carries the metadata of the page it came from, so a query can
be narrowed to one document type *before* the vector search runs.

- embeddings: `BAAI/bge-small-en-v1.5`
- generation: **Mistral-7B-Instruct-v0.3**, 4-bit on a Colab T4 (no OpenAI key needed)
- splitter: `RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)`

The notebook also builds a **page-level index** alongside the sliding-window one and an
**unfiltered retriever** alongside the filtered one, so the two questions in the write-up
(which chunking, how much did filtering help) are answered with measurements.

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Environment

In [ ]:
!pip -q install llama-index-core llama-index-embeddings-huggingface llama-index-llms-huggingface \
                langchain-text-splitters pymupdf gdown "transformers>=4.44" accelerate bitsandbytes
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA: True | Tesla T4


## 2. Prepare the multi-page PDF

Each page gets `page`, `doc_type`, `text`, `doc_id`, `source_file`, `page_in_doc`.
`doc_type` comes from the document title in the page header; a `(continued)` page inherits
the previous page's type, and two same-titled pages with disjoint lot numbers are two
different documents — so `doc_id` counts real document instances, not page runs.

In [ ]:
import os, re, json, gdown
from collections import Counter
import pymupdf

SOURCE_FILE = "pharma-blob-sample.pdf"
if not os.path.exists(SOURCE_FILE):
    gdown.download(id="1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw", output=SOURCE_FILE, quiet=True)
print(SOURCE_FILE, os.path.getsize(SOURCE_FILE), "bytes")

pharma-blob-sample.pdf 12768 bytes


In [ ]:
DOC_TYPE_SLUGS = {
    "Cover Letter": "cover_letter",
    "Certificate of Quality": "certificate_of_quality",
    "Packaging Specification": "packaging_specification",
    "BSE/TSE Declaration": "bse_tse_declaration",
    "Material Description Sheet": "material_description",
    "Supplier Qualification Record": "supplier_qualification",
    "Chain of Custody": "chain_of_custody",
    "Unknown": "unknown",
}
DOC_TYPES = list(DOC_TYPE_SLUGS)

HEADING_CUES = {
    "certificate of quality": "Certificate of Quality",
    "quality certificate": "Certificate of Quality",
    "packaging component specification": "Packaging Specification",
    "declaration regarding transmissible spongiform": "BSE/TSE Declaration",
    "bse/tse": "BSE/TSE Declaration",
    "animal origin statement": "BSE/TSE Declaration",
    "material description sheet": "Material Description Sheet",
    "supplier qualification record": "Supplier Qualification Record",
    "chain of custody": "Chain of Custody",
    "to whom it may concern": "Cover Letter",
}
CONTINUATION_CUES = ("(continued)", "cont'd", "page 2 of", "page 3 of")
ID_PATTERNS = (
    r"lot number\s*:?\s*([a-z0-9\-]{4,})",
    r"document number\s*:?\s*([a-z0-9\-/]{4,})",
    r"product article number\s*:?\s*([0-9 ]{6,})",
)

def identifiers(text):
    low, ids = text.lower(), set()
    for pat in ID_PATTERNS:
        ids.update(m.group(1).strip() for m in re.finditer(pat, low))
    return ids


def page_doc_type(text):
    head = "\n".join(text.strip().splitlines()[:12]).lower()   # letterheads push the title down
    for k, v in HEADING_CUES.items():
        if k in head:
            return v
    low = text.lower()
    for k, v in HEADING_CUES.items():
        if k in low:
            return v
    return "Unknown"


def extract_pages(path, source_file):
    doc = pymupdf.open(path)
    pages, prev_type, prev_text, page_in_doc, doc_id = [], None, None, -1, None
    counters = Counter()
    for i, pg in enumerate(doc):
        text = pg.get_text("text").strip()
        dtype = page_doc_type(text)
        head = "\n".join(text.strip().splitlines()[:4]).lower()
        is_cont = any(c in head for c in CONTINUATION_CUES) and prev_type is not None
        if is_cont:
            dtype = prev_type
        new_doc = (not is_cont) and (
            prev_type is None
            or dtype != prev_type
            or (identifiers(text) and identifiers(prev_text)
                and not (identifiers(text) & identifiers(prev_text)))
        )
        if new_doc:
            slug = DOC_TYPE_SLUGS.get(dtype, "unknown")
            counters[slug] += 1
            doc_id = f"{slug}_{counters[slug]:02d}"
            page_in_doc = 0
        else:
            page_in_doc += 1
        pages.append({"page": i, "doc_type": dtype, "text": text, "doc_id": doc_id,
                      "source_file": source_file, "page_in_doc": page_in_doc})
        prev_type, prev_text = dtype, text
    return pages


pages = extract_pages(SOURCE_FILE, SOURCE_FILE)
print(len(pages), "pages |", len({p['doc_id'] for p in pages}), "documents\n")
for p in pages:
    print(f"p{p['page']:>2} | {p['doc_id']:<28} | in-doc {p['page_in_doc']} | {p['doc_type']}")
print("\nexample record:")
print(json.dumps({**pages[3], "text": pages[3]["text"][:80] + "..."}, indent=2))

10 pages | 8 documents

p 0 | cover_letter_01              | in-doc 0 | Cover Letter
p 1 | certificate_of_quality_01    | in-doc 0 | Certificate of Quality
p 2 | certificate_of_quality_02    | in-doc 0 | Certificate of Quality
p 3 | packaging_specification_01   | in-doc 0 | Packaging Specification
p 4 | packaging_specification_01   | in-doc 1 | Packaging Specification
p 5 | bse_tse_declaration_01       | in-doc 0 | BSE/TSE Declaration
p 6 | material_description_01      | in-doc 0 | Material Description Sheet
p 7 | supplier_qualification_01    | in-doc 0 | Supplier Qualification Record
p 8 | supplier_qualification_01    | in-doc 1 | Supplier Qualification Record
p 9 | chain_of_custody_01          | in-doc 0 | Chain of Custody

example record:
{
  "page": 3,
  "doc_type": "Packaging Specification",
  "text": "Packaging Component Specification\nDocument Number:\nPKG-SPEC-2023-0847\nRevision:\n...",
  "doc_id": "packaging_specification_01",
  "source_file": "pharma-blob-sample.pdf",
  "pag

## 3. Reconstruct the document and chunk it

Pages are concatenated into one logical contract, then split with a sliding window. Because
the split runs over the joined text, a chunk can straddle a page boundary — so each chunk is
mapped back to the page holding its **midpoint**, and inherits that page's metadata.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

PAGE_SEP = "\n\n"

def reconstruct(pages):
    parts, spans, cursor = [], [], 0
    for p in pages:
        parts.append(p["text"])
        spans.append((cursor, cursor + len(p["text"]), p))
        cursor += len(p["text"]) + len(PAGE_SEP)
    return PAGE_SEP.join(parts), spans


def page_for_offset(spans, off):
    for s, e, p in spans:
        if s <= off <= e:
            return p
    return spans[-1][2]


full_contract_text, spans = reconstruct(pages)
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
chunks = splitter.split_text(full_contract_text)

chunk_records, cursor, unlocated = [], 0, 0
for i, ch in enumerate(chunks):
    pos = full_contract_text.find(ch, cursor)
    if pos == -1:
        pos, unlocated = full_contract_text.find(ch), unlocated + 1
    home = page_for_offset(spans, pos + len(ch) // 2)
    chunk_records.append({
        "text": ch,
        "metadata": {"doc_type": home["doc_type"], "chunk_index": i, "doc_id": home["doc_id"],
                     "source_file": home["source_file"], "page": home["page"],
                     "page_in_doc": home["page_in_doc"]},
    })
    cursor = pos + 1

print(f"{len(full_contract_text)} characters -> {len(chunk_records)} chunks "
      f"({unlocated} could not be located)")
print(Counter(c["metadata"]["doc_type"] for c in chunk_records).most_common())

9705 characters -> 28 chunks (0 could not be located)
[('Certificate of Quality', 6), ('Supplier Qualification Record', 6), ('Packaging Specification', 4), ('Cover Letter', 3), ('BSE/TSE Declaration', 3), ('Material Description Sheet', 3), ('Chain of Custody', 3)]


## 4. Attach metadata to each chunk

Metadata is excluded from the **embedded** text so the vector search cannot cheat by reading
the label it is about to be filtered on — the filter has to earn its keep. `doc_type` and
`page` stay visible to the LLM at synthesis time so the answer can cite them.

In [ ]:
from llama_index.core import Document

def to_documents(records):
    docs = []
    for r in records:
        meta = r["metadata"]
        docs.append(Document(
            text=r["text"],
            metadata=meta,
            excluded_embed_metadata_keys=list(meta.keys()),
            excluded_llm_metadata_keys=["chunk_index", "source_file", "page_in_doc"],
        ))
    return docs


all_documents = to_documents(chunk_records)

# baseline for the write-up: one Document per page, no sliding window
page_documents = to_documents([
    {"text": p["text"],
     "metadata": {"doc_type": p["doc_type"], "chunk_index": p["page"], "doc_id": p["doc_id"],
                  "source_file": p["source_file"], "page": p["page"],
                  "page_in_doc": p["page_in_doc"]}}
    for p in pages
])

print(len(all_documents), "sliding-window documents |", len(page_documents), "page documents")
print(all_documents[7].metadata)

28 sliding-window documents | 10 page documents
{'doc_type': 'Certificate of Quality', 'chunk_index': 7, 'doc_id': 'certificate_of_quality_02', 'source_file': 'pharma-blob-sample.pdf', 'page': 2, 'page_in_doc': 0}


## 5. Embedding model and local LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings, VectorStoreIndex

MODEL_ID = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"   # ungated mirror, same weights

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                                device_map="auto", torch_dtype=torch.float16)
hf_model.eval()

def wrap_inst(text):
    return f"[INST] {text} [/INST]"

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = HuggingFaceLLM(
    model=hf_model,
    tokenizer=tokenizer,
    context_window=4096,
    max_new_tokens=220,
    generate_kwargs={"do_sample": False},
    completion_to_prompt=wrap_inst,
    messages_to_prompt=lambda msgs: wrap_inst("\n".join(str(m.content) for m in msgs)),
)
print("embed + llm ready")

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


embed + llm ready


## 6. Build the indexes

In [ ]:
import time
t0 = time.time()
index = VectorStoreIndex.from_documents(all_documents)          # sliding window
page_index = VectorStoreIndex.from_documents(page_documents)    # one chunk per page
print(f"indexed in {time.time()-t0:.1f}s")

indexed in 0.4s


## 7. Predict the document type for a query

The filter needs a `doc_type` to filter on. Mistral picks one from the closed list; a cue
heuristic covers it if the model returns anything unusable.

In [ ]:
QUERY_CUES = {
    "Certificate of Quality": ["lot", "batch", "test result", "released", "conform",
                               "certificate", "integrity", "expiration"],
    "Packaging Specification": ["packaging", "blister", "carton", "tray", "lid film",
                                "ecn", "revision", "configuration change", "drawing"],
    "BSE/TSE Declaration": ["bse", "tse", "animal origin", "animal", "bovine", "spongiform"],
    "Material Description Sheet": ["material", "resin", "silicone", "sterilization",
                                   "autoclave", "gamma", "compatib", "dimensions"],
    "Supplier Qualification Record": ["supplier", "audit", "iso 9001", "iso 13485",
                                      "qualification", "sub-supplier", "delivery"],
    "Chain of Custody": ["chain of custody", "traceab", "shipped", "manufactured at",
                         "facility", "custody"],
    "Cover Letter": ["cover letter", "who sent", "storage temperature", "transmittal"],
}

def heuristic_query_type(query):
    q = " " + re.sub(r"[^a-z0-9 ]", " ", query.lower()) + " "
    best, score = "Unknown", 0
    for t, cues in QUERY_CUES.items():
        s = sum(len(c.split()) ** 2 for c in cues if c in q)
        if s > score:
            best, score = t, s
    return best


ROUTER_RULES = (
    "You route a question to the document inside a pharmaceutical supplier packet that most "
    "likely answers it. Reply with EXACTLY one label from this list and nothing else:\n"
    + "\n".join("- " + t for t in DOC_TYPES) + "\n\n"
    "Guidance:\n"
    "- lot or batch numbers, test results, release statements, expiry dates -> Certificate of Quality\n"
    "- blister trays, cartons, lid film, drawings, ECN revisions, configuration changes -> Packaging Specification\n"
    "- animal-derived material, bovine or porcine origin, BSE or TSE risk -> BSE/TSE Declaration\n"
    "- materials of construction, resins, sterilization or autoclave compatibility, dimensions -> Material Description Sheet\n"
    "- audits, ISO certificates, approved suppliers, sub-suppliers, delivery performance -> Supplier Qualification Record\n"
    "- where a product was made, shipping route, traceability of a lot -> Chain of Custody\n"
    "- who sent the packet, storage-condition letters, what documents are enclosed -> Cover Letter\n\n"
    "Examples:\n"
    "QUESTION: Does this product contain any bovine derived material?\nLABEL: BSE/TSE Declaration\n"
    "QUESTION: What lot numbers were released and did they pass the flow rate test?\nLABEL: Certificate of Quality\n"
    "QUESTION: When was the supplier last audited?\nLABEL: Supplier Qualification Record\n\n"
    "QUESTION: {q}\nLABEL:"
)

def predict_doc_type(query):
    """LLM picks a label from the closed list; cue heuristic covers an unusable answer."""
    raw = ""
    try:
        raw = str(Settings.llm.complete(ROUTER_RULES.format(q=query))).strip()
    except Exception as e:
        print(f"  [warn] router LLM failed ({type(e).__name__}) - using cues")
    low = raw.lower()
    hits = [(low.index(t.lower()), t) for t in DOC_TYPES if t.lower() in low]
    if hits:
        return min(hits)[1], raw          # earliest label mentioned, not first in the list
    return heuristic_query_type(query), raw


QUERY = "Were there any packaging configuration changes?"
predicted_doc_type, raw = predict_doc_type(QUERY)
print("raw:", repr(raw[:90]))
print("predicted_doc_type:", predicted_doc_type)

raw: 'Packaging Specification'
predicted_doc_type: Packaging Specification


## 8. Retrieve with a metadata filter

In [ ]:
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

def make_retriever(idx, doc_type=None, top_k=4):
    if doc_type is None:
        return idx.as_retriever(similarity_top_k=top_k)
    return idx.as_retriever(
        similarity_top_k=top_k,
        filters=MetadataFilters(filters=[
            MetadataFilter(key="doc_type", value=doc_type, operator=FilterOperator.EQ)
        ]),
    )


retriever = make_retriever(index, predicted_doc_type, top_k=4)
retrieved_nodes = retriever.retrieve(QUERY)

for n in retrieved_nodes:
    m = n.node.metadata
    print(f"score {n.score:.3f} | chunk {m['chunk_index']:>2} | page {m['page']} | {m['doc_type']}")
    print("   ", re.sub(r"\s+", " ", n.node.text)[:110], "\n")

score 0.686 | chunk 12 | page 4 | Packaging Specification
    5. Drawing Changes Drawing DWG-29477427-PKG-R3 supersedes DWG-29477427-PKG-R2. Key changes located on drawing: 

score 0.671 | chunk  9 | page 3 | Packaging Specification
    Packaging Component Specification Document Number: PKG-SPEC-2023-0847 Revision: C Effective Date: 2023-11-08 C 

score 0.654 | chunk 11 | page 4 | Packaging Specification
    Packaging Component Specification (continued) Document Number: PKG-SPEC-2023-0847, Page 2 of 2 4. Configuratio 

score 0.619 | chunk 10 | page 3 | Packaging Specification
    PETG ASTM D1003, min 92% clarity Lid Film Tyvek 1073B Per DuPont specification Secondary Carton Corrugated ECT 



In [ ]:
# same query, no filter - what the filter is actually removing
for n in make_retriever(index, None, top_k=4).retrieve(QUERY):
    m = n.node.metadata
    print(f"score {n.score:.3f} | chunk {m['chunk_index']:>2} | page {m['page']} | {m['doc_type']}")

score 0.686 | chunk 12 | page 4 | Packaging Specification
score 0.671 | chunk  9 | page 3 | Packaging Specification
score 0.654 | chunk 11 | page 4 | Packaging Specification
score 0.649 | chunk 26 | page 9 | Chain of Custody


## 9. Synthesize the final answer

In [ ]:
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode

synthesizer = get_response_synthesizer(response_mode=ResponseMode.COMPACT)
response = synthesizer.synthesize(QUERY, nodes=retrieved_nodes)
print("Answer:", response.response.strip())

Answer: Yes, there have been packaging configuration changes. The changes are detailed in the Packaging Specification document with ID packaging_specification_01. Specifically, the blister material for the AKTA ready Flow Kit Packaging Assembly has been changed from PVC to PETG (as per EU Regulation 2023/1297), and there have been updates to the tray cavity dimensions, lid film sealing parameters, and label placement. These changes are reflected in the drawing DWG-29477427-PKG-R3, which supersedes DWG-29477427-PKG-R2. The effective date for these changes is 2023-11-08.


## 10. Final output

In [ ]:
def route_and_answer(query, top_k=4, preview=260):
    dtype, _ = predict_doc_type(query)
    nodes = make_retriever(index, dtype, top_k=top_k).retrieve(query)
    if not nodes:                                  # nothing of that type -> widen
        dtype_used, nodes = None, make_retriever(index, None, top_k=top_k).retrieve(query)
    ans = get_response_synthesizer(response_mode=ResponseMode.COMPACT).synthesize(query, nodes=nodes)
    return {
        "query": query,
        "predicted_doc_type": dtype,
        "matched_chunks": [
            {"text": re.sub(r"\s+", " ", n.node.text)[:preview] + "...",
             "score": round(float(n.score), 4),
             "metadata": n.node.metadata}
            for n in nodes
        ],
        "answer": ans.response.strip(),
    }


result = route_and_answer(QUERY)
print(json.dumps(result, indent=2)[:2600])

{
  "query": "Were there any packaging configuration changes?",
  "predicted_doc_type": "Packaging Specification",
  "matched_chunks": [
    {
      "text": "5. Drawing Changes Drawing DWG-29477427-PKG-R3 supersedes DWG-29477427-PKG-R2. Key changes located on drawing: - Sheet 1: Updated tray cavity dimensions (Section B-B) - Sheet 2: New lid film sealing parameters (Detail C) - Sheet 3: Updated label placement (per...",
      "score": 0.6862,
      "metadata": {
        "doc_type": "Packaging Specification",
        "chunk_index": 12,
        "doc_id": "packaging_specification_01",
        "source_file": "pharma-blob-sample.pdf",
        "page": 4,
        "page_in_doc": 1
      }
    },
    {
      "text": "Packaging Component Specification Document Number: PKG-SPEC-2023-0847 Revision: C Effective Date: 2023-11-08 Component: AKTA ready Flow Kit Packaging Assembly Drawing Reference: DWG-29477427-PKG-R3 1. Scope This specification defines the packaging configuratio...",
      "score": 0

In [ ]:
with open("contract_routing_result.json", "w") as f:
    json.dump(result, f, indent=2)
print("wrote contract_routing_result.json")

wrote contract_routing_result.json


## 11. Measuring the two write-up questions

Six questions, each with the page(s) that actually hold the answer. A configuration counts
as a hit when a retrieved chunk comes from one of those pages; `rank` is the position of the
first such chunk, so lower is better and `-` means it never appeared in the top 4.

In [ ]:
GOLD = [
    ("Were there any packaging configuration changes?",        "Packaging Specification",       [3, 4]),
    ("Which lots passed the pressure integrity test?",         "Certificate of Quality",        [1, 2]),
    ("Does the product contain material of animal origin?",    "BSE/TSE Declaration",           [5]),
    ("When was the supplier last audited?",                    "Supplier Qualification Record", [7, 8]),
    ("Can the pump tubing be autoclaved?",                     "Material Description Sheet",    [6]),
    ("Where are the assemblies manufactured and how are they traced?", "Chain of Custody",      [9]),
]

def first_hit(nodes, gold_pages):
    for r, n in enumerate(nodes, start=1):
        if n.node.metadata["page"] in gold_pages:
            return r
    return None

rows = []
for q, gold_type, gold_pages in GOLD:
    pred, _ = predict_doc_type(q)
    configs = {
        "window_nofilter": make_retriever(index, None, 4).retrieve(q),
        "window_filtered": make_retriever(index, pred, 4).retrieve(q),
        "page_filtered":   make_retriever(page_index, pred, 4).retrieve(q),
    }
    row = {"query": q[:44], "gold_type": gold_type, "routed_to": pred,
           "route_ok": pred == gold_type}
    for name, nodes in configs.items():
        r = first_hit(nodes, gold_pages)
        row[name] = r if r else "-"
        row[name + "_off"] = sum(1 for n in nodes if n.node.metadata["page"] not in gold_pages)
    rows.append(row)

import pandas as pd
res = pd.DataFrame(rows)
for c in ["window_nofilter", "window_filtered", "page_filtered"]:
    hits = sum(1 for v in res[c] if v != "-")
    noise = res[c + "_off"].mean()
    print(f"{c:<16} hit@4 {hits}/{len(res)} | off-target chunks per query {noise:.2f}")
print(f"query routing accuracy {res['route_ok'].sum()}/{len(res)}\n")
print(res[["gold_type", "routed_to", "route_ok", "window_nofilter", "window_filtered", "page_filtered"]].to_string(index=False))
res

window_nofilter  hit@4 6/6 | off-target chunks per query 1.50
window_filtered  hit@4 6/6 | off-target chunks per query 0.00
page_filtered    hit@4 6/6 | off-target chunks per query 0.00
query routing accuracy 6/6

                    gold_type                     routed_to  route_ok  window_nofilter  window_filtered  page_filtered
      Packaging Specification       Packaging Specification      True                1                1              1
       Certificate of Quality        Certificate of Quality      True                1                1              1
          BSE/TSE Declaration           BSE/TSE Declaration      True                1                1              1
Supplier Qualification Record Supplier Qualification Record      True                1                1              1
   Material Description Sheet    Material Description Sheet      True                1                1              1
             Chain of Custody              Chain of Custody      True   

,query,gold_type,routed_to,route_ok,window_nofilter,window_nofilter_off,window_filtered,window_filtered_off,page_filtered,page_filtered_off
0,Were there any packaging configuration chang,Packaging Specification,Packaging Specification,True,1,1,1,0,1,0
1,Which lots passed the pressure integrity tes,Certificate of Quality,Certificate of Quality,True,1,0,1,0,1,0
2,Does the product contain material of animal,BSE/TSE Declaration,BSE/TSE Declaration,True,1,2,1,0,1,0
3,When was the supplier last audited?,Supplier Qualification Record,Supplier Qualification Record,True,1,1,1,0,1,0
4,Can the pump tubing be autoclaved?,Material Description Sheet,Material Description Sheet,True,1,3,1,0,1,0
5,Where are the assemblies manufactured and ho,Chain of Custody,Chain of Custody,True,1,2,1,0,1,0


In [ ]:
# a term-lookup probe: does the index find an exact identifier, the way a contract search must?
for term in ["ECN-2023-0847", "18356721", "DWG-29477427-PKG-R3", "BSE-RA-2023-AKTA-001"]:
    nodes = make_retriever(index, None, 3).retrieve(term)
    found = [n.node.metadata["chunk_index"] for n in nodes if term.lower() in n.node.text.lower()]
    print(f"{term:<22} literal chunk in top 3: {found or 'NO'}")

ECN-2023-0847          literal chunk in top 3: [11]
18356721               literal chunk in top 3: NO
DWG-29477427-PKG-R3    literal chunk in top 3: [12, 11]
BSE-RA-2023-AKTA-001   literal chunk in top 3: [15, 14]


## 12. Notes

**Measured on this run** (6 questions, top-k = 4, gold = the page that actually holds the answer):

| configuration | hit@4 | off-target chunks per query |
|---|---|---|
| sliding window, no filter | 6/6 | 1.50 |
| sliding window, `doc_type` filter | 6/6 | 0.00 |
| one chunk per page, `doc_type` filter | 6/6 | 0.00 |

Query routing accuracy 6/6; in every configuration the first correct chunk came back at rank 1.

- **Filtering bought precision, not recall.** Dense search alone already found the right
  page for all six questions, but it padded the top 4 with 1.5 off-target chunks on average —
  the packaging question pulled in a Chain of Custody chunk at rank 4. With the filter on,
  every returned chunk came from the right document and nothing was lost.
- **The filter is only as good as the router.** A first pass used a bare one-line routing
  prompt, scored 3/6 on query classification, and dragged filtered hit@4 down to 3/6 — when
  the router picks the wrong type, the filter confidently excludes the answer. Rewriting the
  prompt with an explicit cue-to-document mapping and three examples took routing to 6/6.
  Unfiltered search degrades gracefully; filtered search fails hard.
- **Chunk metadata only helps if it is per page, not per file.** Tagging all 28 chunks
  `Packaging Specification` — as the task template does — would make the filter a no-op,
  since every chunk would pass it. Mapping each chunk back to the page holding its midpoint
  is what turns `doc_type` into a real narrowing key.
- **Metadata is excluded from the embedded text** so the vector search cannot read the label
  it is about to be filtered on. Without that, the unfiltered baseline looks better than it is.
- **Exact identifiers are the weak spot.** The probe cell finds ECN-2023-0847,
  DWG-29477427-PKG-R3 and BSE-RA-2023-AKTA-001 in the top 3, but misses lot number 18356721
  entirely — a bare digit string carries almost no semantic signal. That is the evidence
  behind the "would you trust this on a legal contract" answer: not on its own, not without
  a lexical (BM25) leg beside the dense one.